[![img/pythonista.png](img/pythonista.png)](https://www.pythonista.io)

# Proporciones, variables categóricas y más de dos grupos.

**py401 · EDA Práctico** · Pythonista® · Instructor: José Luis Chiquete · Módulo III: *Inferencia estadística*

> 📝 **Versión del alumno.** Las celdas marcadas con `# TU CÓDIGO AQUÍ` y las de *Tu respuesta* son para ti.

En este *notebook* se probarán proporciones y la independencia de dos variables categóricas con la prueba *z* y la chi cuadrada, y se compararán más de dos grupos con ANOVA, Levene, Welch y Tukey.

**Al terminar podrás:**

* comparar dos proporciones con su diferencia, su IC, el riesgo relativo y la prueba *z*;
* medir la asociación entre dos variables de categorías con la chi cuadrada y la V de Cramér;
* comparar tres o más grupos con el ANOVA de Welch (una H0 y una H1) y localizar los pares que difieren con Tukey o con Holm;
* explicar por qué muchas pruebas por pares inflan las falsas alarmas y por qué, con muestras enormes, significativo no es lo mismo que importante.

## 🔁 Antes de empezar: repaso de memoria.

Sin abrir `20_pruebas_para_dos_grupos.ipynb`, escribe con tus palabras, en tres o cuatro frases, la idea clave de ese *notebook*: cómo se comparan dos grupos: qué prueba se usa por defecto para comparar medias, cuándo conviene otra y qué cifras acompañan siempre al p-value. No es un cuestionario ni hay una sola respuesta correcta: se trata de reconstruir la idea de memoria, porque recordar sin mirar ayuda a fijarla. Cuando termines, compáralo con el resumen que está al inicio de la versión con soluciones de este *notebook* y corrige lo que haga falta.

**✍️ Tu respuesta:**

_(escribe aquí)_

## 🧭 Cómo usar este *notebook*.

* Ejecuta las celdas **en orden** (`Shift + Enter`).
* **Ejercicio básico:** lo que todos deben dominar.
* **Reto (opcional):** para quien quiere profundizar. Se puede resolver como práctica.
* Las celdas `# TU CÓDIGO AQUÍ` y *✍️ Tu respuesta* son para ti.

## 📚 Referencias.

py311 no cubre inferencia, así que la referencia de la API es la documentación oficial: [*SciPy* · `scipy.stats`](https://docs.scipy.org/doc/scipy/reference/stats.html) (`ttest_ind`, `mannwhitneyu`, `permutation_test`, `chi2_contingency`, `f_oneway`, `levene`), [*statsmodels* · pruebas de proporciones](https://www.statsmodels.org/stable/generated/statsmodels.stats.proportion.proportions_ztest.html), [*statsmodels* · `pairwise_tukeyhsd`](https://www.statsmodels.org/stable/generated/statsmodels.stats.multicomp.pairwise_tukeyhsd.html) y [*statsmodels* · `anova_oneway`](https://www.statsmodels.org/stable/generated/statsmodels.stats.oneway.anova_oneway.html).

### Preparación del entorno.

* La siguiente celda importará las bibliotecas y las funciones de *SciPy* y *statsmodels* que se usarán, y definirá `DATA` y la semilla `rng`.

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA = next(os.path.join(p, "data") for p in (os.curdir, os.pardir)
            if os.path.isfile(os.path.join(p, "data", "penguins.csv")))
pd.set_option("display.max_columns", 20)
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(401)
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.oneway import anova_oneway
from statsmodels.stats.proportion import proportions_ztest

* La siguiente celda leerá `penguins_limpio.csv` (o lo regenerará si no existe), `titanic.csv` y `tips.csv`, y creará `tip_pct`.

In [ ]:
ruta_limpio = os.path.join(DATA, "penguins_limpio.csv")
medidas = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
if not os.path.isfile(ruta_limpio):
    (
        pd.read_csv(os.path.join(DATA, "penguins.csv"))
        .assign(sex=lambda d: d["sex"].str.title())
        .dropna(subset=medidas, how="all")
        .to_csv(ruta_limpio, index=False)
    )
pinguinos = pd.read_csv(ruta_limpio)
titanic = pd.read_csv(os.path.join(DATA, "titanic.csv"))
propinas = pd.read_csv(os.path.join(DATA, "tips.csv")).assign(tip_pct=lambda d: 100 * d["tip"] / d["total_bill"])
pinguinos.shape, titanic.shape, propinas.shape

### Objetos de los *notebooks* anteriores.

Este *notebook* es independiente: la siguiente celda vuelve a crear lo que se necesita de los *notebooks* anteriores.

* La siguiente celda definirá `d_cohen`, presentada en `20_pruebas_para_dos_grupos.ipynb`.

In [ ]:
def d_cohen(a, b):
    a, b = np.asarray(a), np.asarray(b)
    varianza_combinada = ((len(a) - 1) * a.var(ddof=1) + (len(b) - 1) * b.var(ddof=1)) / (len(a) + len(b) - 2)
    return (a.mean() - b.mean()) / np.sqrt(varianza_combinada)

---
## 1. Proporciones y variables categóricas.

* **Prueba *z* de dos proporciones** (`proportions_ztest(conteos, totales)`): ¿difiere la proporción de éxito entre dos grupos?
* **Chi cuadrada de independencia** (`stats.chi2_contingency(tabla)`): ¿están asociadas dos variables categóricas? Compara la tabla observada con la **esperada** si fueran independientes.
* **Supuestos clave:** observaciones independientes (cada persona aparece una sola vez) y frecuencias **esperadas** suficientemente grandes (en general ≥ 5 en cada celda; para la prueba *z*, que haya bastantes éxitos y bastantes fracasos en cada grupo). Si no se cumple, se agrupan categorías o se usa la prueba exacta de Fisher (`stats.fisher_exact`, para tablas 2×2).

**Concepto: prueba *z* de dos proporciones.** Si H0 fuera cierta (los dos grupos tienen la misma proporción), se juntan los dos grupos para estimar esa proporción común. Luego se mide la diferencia entre las dos proporciones observadas **en errores estándar**, igual que la *t* con las medias. Por el TCL, ese número se compara con la campana normal: más allá de unos 2 errores estándar, la diferencia es difícil de atribuir al azar.

**Ejemplo:** si 30% de los clientes de una sucursal y 35% de otra recomiendan el servicio, la prueba responde si 5 puntos de diferencia son mucho o poco **para el número de clientes encuestados**.

* La siguiente celda comparará con la prueba *z* la tasa de supervivencia de mujeres y hombres en el Titanic.

In [ ]:
tabla_sexo = pd.crosstab(titanic["sex"], titanic["survived"])
sobrevivientes = tabla_sexo[1].to_numpy()
totales = tabla_sexo.sum(axis=1).to_numpy()
estadistico_z, valor_p_z = proportions_ztest(sobrevivientes, totales)
(sobrevivientes / totales).round(3), round(float(estadistico_z), 2), float(valor_p_z)

La diferencia es enorme (74.2% contra 18.9%) y su valor *p* es diminuto (del orden de 10⁻⁵⁹; nunca es exactamente 0). Para reportar la magnitud se usa la **diferencia de proporciones** con su IC (como en `18_bootstrap_e_interpretacion_de_intervalos.ipynb`) o el **riesgo relativo**: la proporción de mujeres que sobrevivió fue casi 4 veces la de los hombres (74.2 / 18.9 ≈ 3.9). Ojo: los pasajeros del Titanic no son una muestra al azar de nada, así que estas cifras describen a esos pasajeros y no se pueden generalizar a otros viajes.

**Ejemplo:** ¿están asociadas la clase y la supervivencia? Se revisa la tabla esperada antes de interpretar.

**Concepto: chi cuadrada.** Primero se construye la tabla **esperada**: cuántos casos habría en cada celda si las dos variables no tuvieran relación. Para cada celda, se multiplica el total de su fila por el total de su columna y se divide entre el total general. Luego la chi cuadrada suma, celda por celda, qué tan lejos está lo observado de lo esperado (en relación con lo esperado). Si las variables no tienen relación, la suma queda chica.

**Ejemplo:** si 38% de todos los pasajeros sobrevivió y había 216 en primera clase, sin relación entre clase y supervivencia se esperarían unos 83 sobrevivientes de primera (216 × 0.384). Si se observan muchos más, eso aporta mucho a la chi cuadrada.

Su tamaño del efecto es la **V de Cramér**, que va de 0 (sin relación) a 1 (relación perfecta).

* La siguiente celda aplicará la chi cuadrada de independencia a `class` contra `survived` y mostrará la tabla esperada.

In [ ]:
tabla_clase = pd.crosstab(titanic["class"], titanic["survived"])
chi2 = stats.chi2_contingency(tabla_clase)
esperada = pd.DataFrame(chi2.expected_freq, index=tabla_clase.index, columns=tabla_clase.columns).round(1)
round(float(chi2.statistic), 1), int(chi2.dof), float(chi2.pvalue), esperada

Todas las frecuencias esperadas superan 5, así que la aproximación es válida. La chi cuadrada dice que hay asociación, pero no cuánta: el tamaño del efecto es la **V de Cramér** (0 = independencia, 1 = asociación perfecta; no es un porcentaje), que se calcula con `stats.contingency.association(tabla, method="cramer")`.

| V de Cramér (orientativo, para tablas con solo 2 filas o 2 columnas) | 0.1 | 0.3 | 0.5 |
|---|---|---|---|
| Magnitud | Pequeña | Mediana | Grande |

📖 **Para profundizar (opcional).** *Las fórmulas de la sección 1.*
* Prueba *z* de dos proporciones: bajo H0 ($p_A = p_B$) se estima la proporción común $\hat{p}$ con los dos grupos juntos y $$z = \frac{\hat{p}_A - \hat{p}_B}{\sqrt{\hat{p}(1-\hat{p})\left(\frac{1}{n_A} + \frac{1}{n_B}\right)}}$$
* Chi cuadrada: la frecuencia esperada de cada celda es $E_{ij} = \dfrac{(\text{total de la fila } i)(\text{total de la columna } j)}{n}$ y $$\chi^2 = \sum_{i,j} \frac{(O_{ij} - E_{ij})^2}{E_{ij}}, \qquad g.l. = (f - 1)(c - 1)$$
* V de Cramér: $V = \sqrt{\dfrac{\chi^2}{n\,(\min(f, c) - 1)}}$.

**💡 Ideas clave de la sección 1:**
* Prueba *z* para dos proporciones; chi cuadrada para dos categóricas, revisando que las frecuencias esperadas sean ≥ 5.
* La magnitud se reporta con la diferencia de proporciones, el riesgo relativo o la V de Cramér.

### Ejercicio 1 · Mujeres y hombres en tercera clase.

**🎯 Objetivo:** aplicar la prueba *z* y la V de Cramér dentro de un subgrupo. **Qué se espera aprender:** a revisar si una asociación global se mantiene al controlar por otra variable.

¿La ventaja de las mujeres se mantiene en tercera clase? Con los pasajeros de `class == "Third"`, compara la supervivencia de mujeres y hombres con la prueba *z* y calcula la V de Cramér de la tabla 2×2.

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ

**✍️ Tu respuesta:**

_(escribe aquí)_

---
## 2. Más de dos grupos.

Comparar tres grupos con tres pruebas *t* infla el error tipo I (sección 2 de `19_logica_del_contraste.ipynb`). El procedimiento recomendado es:

1. **Describir los grupos:** cuántos datos tiene cada uno, su media y su desviación estándar. **Levene** (`stats.levene`) sirve para **describir** si la dispersión es parecida entre grupos; en *SciPy* mide, por omisión, la distancia de cada dato a la **mediana** de su grupo (`center="median"`). No se usa como interruptor para elegir la prueba: con grupos chicos casi nunca rechaza aunque las dispersiones difieran, y con grupos grandes rechaza por diferencias sin importancia.
2. **ANOVA de Welch de entrada** (`anova_oneway(..., use_var="unequal")` de *statsmodels*): H0 = todas las medias son iguales. Igual que la prueba de Welch para dos grupos, no supone que los grupos tengan la misma dispersión. Si las dispersiones sí son parecidas, da casi lo mismo que el **ANOVA clásico** (`stats.f_oneway`), que sí supone varianzas iguales. Los dos suponen observaciones independientes y datos de cada grupo sin una asimetría extrema (con grupos de unas decenas de datos, el TCL ayuda).
3. Si se rechaza H0, **post hoc de Tukey** (`pairwise_tukeyhsd`) para saber **qué** pares difieren, controlando el error de las comparaciones múltiples. Tukey supone dispersiones parecidas; si no lo son, una alternativa es comparar cada par con Welch y corregir los valores *p* con Holm.
4. Tamaño del efecto: **η²** = suma de cuadrados entre grupos / suma de cuadrados total (proporción de la variabilidad que se asocia con el grupo).

**¿Por qué no varias pruebas *t*?** Con 3 grupos hay 3 pares; si cada comparación admite 5% de falsas alarmas, la probabilidad de al menos una sube hasta 14% (1 − 0.95³) si las tres pruebas fueran independientes. En una **simulación** (datos simulados: 10,000 repeticiones con 3 grupos de 30 datos cada uno y sin ninguna diferencia real entre ellos), la probabilidad de al menos una falsa alarma sale alrededor de 12%. Con 6 grupos son 15 pares: la cuenta con independencia da 54% (1 − 0.95¹⁵), y la misma simulación con 6 grupos da alrededor de 36%. Las pruebas de pares comparten datos, por eso la probabilidad real queda por debajo de la cuenta con independencia.

**Tukey, en palabras.** Cuando el ANOVA dice que algún grupo difiere, Tukey compara todos los pares de grupos a la vez, con un umbral ajustado para que la probabilidad de un falso hallazgo en todo el conjunto siga en 5%. Entre los pingüinos, dice que los Gentoo difieren de los Adelie y de los Chinstrap (unos 1,375 y 1,343 g más), pero no distingue a los Adelie de los Chinstrap (32 g de diferencia, *p* = 0.88). Más adelante, en esta misma sección, se calculan con código.

**Kruskal-Wallis** es el ANOVA por posiciones: compara si un grupo tiende a tener valores más altos que los demás y no pide una forma de campana.

**Concepto: ANOVA y η².** La variabilidad total de los datos se separa en dos partes: la que hay **entre** grupos (qué tan distintas son las medias de los grupos) y la que hay **dentro** de los grupos (qué tan distintos son los individuos de un mismo grupo).

El estadístico *F* compara ambas partes: si los grupos no difieren, las dos son parecidas y *F* queda cerca de 1; si las medias de los grupos están muy separadas en comparación con la variación dentro de cada uno, *F* crece. El tamaño del efecto, **η²**, es la fracción de la variabilidad total que se asocia con el grupo: un η² de 0.7 quiere decir que 70% de la variación entre individuos se asocia con el grupo al que pertenecen (no dice que el grupo sea la causa).

**Ejemplo:** en la masa de los pingüinos, la especie se asocia con alrededor de 67% de la variación de la masa entre individuos (η² ≈ 0.67), porque los Gentoo pesan mucho más. En cambio, en el ejemplo de esta sección el día de la semana casi no se asocia con la propina porcentual.

* La siguiente celda definirá `eta_cuadrada` a partir de los valores y las etiquetas de grupo.

In [ ]:
def eta_cuadrada(valores, grupos):
    datos = pd.DataFrame({"valor": valores, "grupo": grupos}).dropna()
    media_global = datos["valor"].mean()
    por_grupo = datos.groupby("grupo")["valor"].agg(["count", "mean"])
    suma_entre = (por_grupo["count"] * (por_grupo["mean"] - media_global) ** 2).sum()
    suma_total = ((datos["valor"] - media_global) ** 2).sum()
    return suma_entre / suma_total

**Ejemplo:** se usará `tip_pct` por día de la semana en `tips`, un caso donde **no** se espera una diferencia grande.

* La siguiente celda calculará Levene (como descripción de la dispersión), el ANOVA clásico, el ANOVA de Welch y Kruskal-Wallis (alternativa por posiciones) para `tip_pct` por día, y calculará η².

In [ ]:
grupos_dia = [grupo["tip_pct"].to_numpy() for _, grupo in propinas.groupby("day")]
welch_dia = anova_oneway(propinas["tip_pct"], propinas["day"], use_var="unequal")
pd.DataFrame({
    "valor p": [stats.levene(*grupos_dia).pvalue, stats.f_oneway(*grupos_dia).pvalue,
                welch_dia.pvalue, stats.kruskal(*grupos_dia).pvalue],
}, index=["Levene", "ANOVA", "ANOVA de Welch", "Kruskal-Wallis"]).round(3), round(float(eta_cuadrada(propinas["tip_pct"], propinas["day"])), 3)

Ninguna prueba encuentra diferencias y η² ≈ 0.01: el día se asocia con apenas alrededor de 1% de la variabilidad de la propina porcentual, que es más o menos lo que se esperaría solo por azar con 4 grupos y 244 mesas (≈ 0.012). Como el ANOVA no rechaza, aquí no se sigue con Tukey; por convención se hace después de un ANOVA que rechaza (aunque Tukey ya controla por sí solo el riesgo de algún falso hallazgo y, si se corre, tampoco encuentra pares: su menor *p* es 0.48). Lo que sí es *p-hacking* es comparar muchos pares con pruebas *t* sin corrección y quedarse con los que salen significativos.

| η² (orientativo) | 0.01 | 0.06 | 0.14 |
|---|---|---|---|
| Magnitud | Pequeño | Mediano | Grande |

**Ejemplo con diferencias grandes: la masa corporal por especie.** En las propinas no había nada que encontrar. Ahora se repite el procedimiento con `body_mass_g` de los pingüinos, donde ya se vio que los Gentoo pesan mucho más. Son datos reales observados, no de un experimento: las cifras describen cómo se asocian la especie y la masa en estos 342 pingüinos.

* La siguiente celda resumirá la masa por especie (cuántos pingüinos, media y desviación estándar) y calculará Levene (como descripción), el ANOVA clásico, el ANOVA de Welch, Kruskal-Wallis y η².

In [ ]:
masa = pinguinos.dropna(subset=["body_mass_g"])
grupos_masa = [grupo["body_mass_g"].to_numpy() for _, grupo in masa.groupby("species")]
resumen_masa = masa.groupby("species")["body_mass_g"].agg(["count", "mean", "std"]).round(1)
levene_masa = stats.levene(*grupos_masa)
anova_masa = stats.f_oneway(*grupos_masa)
welch_masa = anova_oneway(masa["body_mass_g"], masa["species"], use_var="unequal")
kruskal_masa = stats.kruskal(*grupos_masa)
pruebas_masa = pd.DataFrame({
    "estadístico": [levene_masa.statistic, anova_masa.statistic, welch_masa.statistic, kruskal_masa.statistic],
    "valor p": [levene_masa.pvalue, anova_masa.pvalue, welch_masa.pvalue, kruskal_masa.pvalue],
}, index=["Levene (descriptivo)", "ANOVA clásico", "ANOVA de Welch", "Kruskal-Wallis"])
(resumen_masa, pruebas_masa.round({"estadístico": 1}), round(float(welch_masa.df_denom), 1),
 round(float(eta_cuadrada(masa["body_mass_g"], masa["species"])), 2))

**Cómo leer el resultado.**

* **Resumen:** los Adelie (151) y los Chinstrap (68) pesan en promedio unos 3,700 g; los Gentoo (123), unos 5,076 g. Las desviaciones estándar van de 384 a 504 g: la mayor es apenas 1.3 veces la menor.
* **Levene** da *p* = 0.006: con 342 pingüinos detecta que la dispersión no es idéntica entre especies, aunque la diferencia es moderada. Es una **descripción**, no una orden: no cambia la prueba, porque el ANOVA de Welch ya se eligió de entrada.
* **ANOVA de Welch:** *F* ≈ 317.6, con 2 y 189.5 grados de libertad (Welch ajusta el segundo número por las dispersiones distintas), y *p* < 0.001. Se **rechaza** H0: no todas las especies tienen la misma masa media. El ANOVA clásico (*F* ≈ 343.6, *p* < 0.001) llega a la misma conclusión.
* **Kruskal-Wallis:** *H* ≈ 217.6, *p* < 0.001. Por posiciones, también hay al menos una especie que tiende a pesar más.
* **η² ≈ 0.67:** la especie se asocia con alrededor de 67% de la variación de la masa entre individuos, un efecto grande. Solo por azar, con 3 grupos y 342 pingüinos, se esperaría alrededor de 0.006.

Los valores *p* tan pequeños aparecen en notación científica (por ejemplo, `3.09e-61` quiere decir 0.000…0309, con 60 ceros después del punto) y se reportan como "*p* < 0.001". Que el valor *p* sea diminuto no dice cuánto pesan de más los Gentoo: eso lo dicen las medias, el η² y los intervalos de Tukey.

**Tukey con código.** El ANOVA solo dice que *alguna* especie difiere. Tukey dice **cuáles** pares difieren y por cuánto, con un intervalo para cada diferencia.

Su margen se arma como el de cualquier intervalo (`17_error_estandar_e_intervalos_de_confianza.ipynb`): un número de una tabla multiplicado por el **error estándar de la diferencia** entre las dos medias. Lo que cambia es el número de la tabla. Una prueba suelta usaría alrededor de 1.97; Tukey usa uno más grande para que el 95% valga para los tres pares a la vez. Ese número es el *q* de la tabla de Tukey **dividido entre la raíz de 2**, porque la tabla de Tukey está hecha con el error estándar de una sola media y aquí se usa el de la diferencia de dos medias.

* La siguiente celda aplicará `pairwise_tukeyhsd` a la masa por especie y, para el par Gentoo–Adelie, reconstruirá el margen paso a paso.

In [ ]:
tukey_masa = pairwise_tukeyhsd(masa["body_mass_g"], masa["species"], alpha=0.05)
print(tukey_masa.summary())

# El margen de Tukey para Gentoo contra Adelie, paso a paso.
k, n_total = len(grupos_masa), len(masa)
varianza_dentro = sum(((g - g.mean()) ** 2).sum() for g in grupos_masa) / (n_total - k)
adelie = masa.loc[masa["species"] == "Adelie", "body_mass_g"]
gentoo = masa.loc[masa["species"] == "Gentoo", "body_mass_g"]
ee_diferencia = np.sqrt(varianza_dentro * (1 / len(adelie) + 1 / len(gentoo)))
q = stats.studentized_range.ppf(0.95, k, n_total - k)
pd.Series({
    "diferencia (g)": gentoo.mean() - adelie.mean(),
    "EE de la diferencia (g)": ee_diferencia,
    "q de Tukey": q,
    "q / raíz de 2": q / np.sqrt(2),
    "margen de Tukey (g)": q / np.sqrt(2) * ee_diferencia,
    "margen de una sola prueba (g)": stats.t.ppf(0.975, n_total - k) * ee_diferencia,
}).round(2)

**Cómo leer el resultado.**

* En la tabla, `meandiff` es la diferencia de medias (el segundo grupo menos el primero), `lower` y `upper` son los extremos del intervalo, `p-adj` es el valor *p* ya ajustado por las tres comparaciones y `reject` indica si se rechaza que esa diferencia sea 0. Un `p-adj` de `0.0` no es exactamente 0: es un valor tan pequeño que se redondea, y se reporta como "*p* < 0.001".
* **Gentoo – Adelie:** 1,375 g más, con intervalo de 1,243 a 1,508 g. **Gentoo – Chinstrap:** 1,343 g más, de 1,178 a 1,507 g. En los dos pares se rechaza H0.
* **Chinstrap – Adelie:** 32 g, con intervalo de −127 a 191 g y *p* = 0.88. El intervalo incluye el 0: **no se rechaza** que pesen lo mismo. Eso no prueba que pesen igual; solo dice que con estos datos no se distinguen.
* **El margen, paso a paso:** el error estándar de la diferencia Gentoo–Adelie es 56.1 g. El *q* de la tabla es 3.33 y, dividido entre la raíz de 2, queda 2.35. El margen de Tukey es 2.35 × 56.1 ≈ 132 g, así que el intervalo va de 1,375 − 132 a 1,375 + 132. Una prueba suelta, que no protege a las otras dos comparaciones, usaría 1.97 × 56.1 ≈ 110 g: Tukey ensancha el intervalo como precio por hacer tres comparaciones.
* El 95% es del **procedimiento**: si se repitiera el muestreo muchas veces, en alrededor de 95% de las repeticiones los tres intervalos juntos atraparían las tres diferencias verdaderas. No es la certeza de que estos intervalos en particular las contengan.

**Otra forma: Welch por pares con Bonferroni o Holm.** Tukey supone que los grupos tienen dispersiones parecidas. Si no se quiere suponer eso, se compara cada par con la prueba de Welch y se corrigen los tres valores *p*:

* **Bonferroni** exige a cada uno de los 3 pares un *p* menor que 0.05 / 3 ≈ 0.017.
* **Holm** ordena los *p* de menor a mayor y exige 0.017 al más chico, 0.025 (0.05 / 2) al segundo y 0.05 al tercero; se detiene en el primero que no cumple. Nunca rechaza menos que Bonferroni y a veces rechaza más.

* La siguiente celda comparará los tres pares con Welch y ajustará los valores *p* con Bonferroni y con Holm (`multipletests`).

In [ ]:
pares = [("Gentoo", "Adelie"), ("Gentoo", "Chinstrap"), ("Chinstrap", "Adelie")]
por_especie = {especie: grupo["body_mass_g"] for especie, grupo in masa.groupby("species")}
pares_masa = pd.DataFrame({
    "par": [f"{a} – {b}" for a, b in pares],
    "p": [stats.ttest_ind(por_especie[a], por_especie[b], equal_var=False).pvalue for a, b in pares],
})
pares_masa["p Bonferroni"] = multipletests(pares_masa["p"], method="bonferroni")[1]
pares_masa["p Holm"] = multipletests(pares_masa["p"], method="holm")[1]
pares_masa["rechaza con Bonferroni"] = multipletests(pares_masa["p"], method="bonferroni")[0]
pares_masa["rechaza con Holm"] = multipletests(pares_masa["p"], method="holm")[0]
pares_masa

**Cómo leer el resultado.** Los *p* de Welch son del orden de 10⁻⁶⁵ (Gentoo – Adelie), 10⁻⁴⁸ (Gentoo – Chinstrap) y 0.59 (Chinstrap – Adelie). Las columnas `p Bonferroni` y `p Holm` son los *p* ya ajustados, que se comparan directamente con 0.05. Con las dos correcciones se rechaza H0 en los dos pares con Gentoo y no se rechaza en Chinstrap – Adelie: la misma conclusión que Tukey.

Aquí Bonferroni y Holm coinciden porque dos *p* son diminutos y el tercero está lejos de 0.05. **Ejemplo hipotético** (no son estos datos): si el tercer *p* hubiera sido 0.03, Bonferroni no rechazaría (0.03 es mayor que 0.017), pero Holm sí (al tercero solo le exige 0.05). Por eso Holm es la corrección recomendada por omisión.

**Consecuencia: significativo no es lo mismo que importante.** Con muestras enormes, cualquier diferencia mínima resulta significativa. Se simularán dos grupos de 100,000 personas cuyas medias difieren 0.45 puntos en una escala de puntaje con desviación estándar de 15.

* La siguiente celda calculará el valor *p* de Welch y la d de Cohen de esa diferencia mínima.

In [ ]:
enorme_a = rng.normal(100, 15, size=100_000)
enorme_b = rng.normal(100.45, 15, size=100_000)
float(stats.ttest_ind(enorme_b, enorme_a, equal_var=False).pvalue), round(float(d_cohen(enorme_b, enorme_a)), 3)

El valor *p* es diminuto, pero la d es ≈ 0.02 a 0.03 (la diferencia verdadera es 0.45 / 15 = 0.03 desviaciones estándar; la de la muestra varía un poco por azar): un efecto **despreciable**. Reportar solo "*p* < 0.001" llevaría a creer que es un hallazgo importante. Por eso el tamaño del efecto y su IC son obligatorios en el informe.

📖 **Para profundizar (opcional).** *Las fórmulas del ANOVA (sección 2).* La suma de cuadrados total se descompone así:

$$SC_{\text{total}} = \sum_{i}(x_i - \bar{x})^2 = \underbrace{\sum_{g} n_g(\bar{x}_g - \bar{x})^2}_{SC_{\text{entre}}} + \underbrace{\sum_{g}\sum_{i \in g}(x_i - \bar{x}_g)^2}_{SC_{\text{dentro}}}$$

Con $k$ grupos y $n$ datos, $F = \dfrac{SC_{\text{entre}} / (k - 1)}{SC_{\text{dentro}} / (n - k)}$ y $\eta^2 = SC_{\text{entre}} / SC_{\text{total}}$.

**💡 Ideas clave de la sección 2:**
* ANOVA de Welch de entrada (Levene solo describe la dispersión) → Tukey (por convención, cuando el ANOVA rechaza) → η².
* Kruskal-Wallis es la alternativa por posiciones.
* Tukey ensancha el margen de cada par para que el 95% valga para todos los pares a la vez; con Welch por pares, Holm es la corrección recomendada (nunca rechaza menos que Bonferroni).
* Con *n* enorme, hasta una diferencia minúscula sale significativa: el tamaño del efecto decide si importa.

### Ejercicio 2 · La longitud del pico en las tres especies.

**🎯 Objetivo:** aplicar, paso a paso, el procedimiento para comparar más de dos grupos: ANOVA de Welch y, si se rechaza H0, comparaciones por pares. **Qué se espera aprender:** a plantear una sola H0 y una sola H1 para tres grupos, a no confundir "al menos una especie difiere" con "todas difieren" y a localizar qué pares son distintos sin inflar las falsas alarmas.

En la sección 2 ya se resolvió la masa por especie. Aquí se usará otra medida de los mismos pingüinos: `bill_length_mm` (la longitud del pico, en milímetros) por especie (`species`). Son los mismos tres grupos, pero las especies que se parecen en la masa no tienen por qué parecerse en el pico, así que el resultado no se puede adivinar con lo que ya se vio. Los datos son observados, no de un experimento: el resultado describe cómo se asocian la especie y el pico en estos pingüinos.

Sigue estos pasos con el *DataFrame* `pinguinos`, que ya está cargado:

1. **Hipótesis.** Escribe en palabras H0 y H1. Recuerda: con tres grupos hay **una** H0 y **una** H1, y la H1 no dice que las tres especies sean distintas entre sí.
2. **Describe los grupos.** Quita los datos faltantes de `bill_length_mm` y, con `groupby("species")`, calcula cuántos pingüinos hay, la media y la desviación estándar de cada especie. Después calcula Levene con `stats.levene` (por omisión usa la mediana de cada grupo, `center="median"`). Úsalo **solo para describir** si la dispersión es parecida: no decide qué prueba se usa.
3. **ANOVA de Welch.** Aplica `anova_oneway(valores, grupos, use_var="unequal")`. Si quieres comparar, calcula también el ANOVA clásico con `stats.f_oneway`. Con un nivel de significancia de 0.05, decide si se **rechaza** o **no se rechaza** H0. Calcula el tamaño del efecto con la función `eta_cuadrada` de la sección 2.
4. **Comparaciones por pares (solo si se rechaza H0).** Usa `pairwise_tukeyhsd` o, como alternativa, compara los tres pares con Welch (`stats.ttest_ind(..., equal_var=False)`) y corrige los tres p-values con Holm (`multipletests(..., method="holm")`). ¿Qué pares difieren y por cuántos milímetros?
5. **Conclusión.** Escribe dos o tres frases en lenguaje simple: qué especie tiene el pico más largo y cuál el más corto, qué pares difieren y por cuánto, y qué tan grande es el efecto. Usa frases como "se asocia con" y no "causa" o "explica".

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ

**✍️ Tu respuesta:**

_(escribe aquí)_

### Reto 1 · Doce pruebas a la vez.

**🎯 Objetivo:** aplicar correcciones por comparaciones múltiples a un caso real. **Qué se espera aprender:** cuándo las correcciones cambian las conclusiones y cuándo no.

Para cada especie y cada una de las cuatro medidas, compara machos y hembras con Welch (12 pruebas). Aplica las correcciones de Bonferroni y de Holm con `multipletests`. ¿Cuántas pruebas siguen siendo significativas? Agrega la d de Cohen de cada comparación: ¿cuál es la diferencia entre sexos más pequeña?

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ (opcional)

**✍️ Tu respuesta:**

_(escribe aquí)_

---
## ✅ Cierre.

**Práctica (en Classroom):** terminar los ejercicios pendientes de este *notebook* y, si es posible, un reto.

**Siguiente *notebook*:** `22_tamano_del_efecto.ipynb`.

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>